# Learning objectives

- Reproduce the portal prompt-agent in the SDK.

# Portal-to-SDK mapping

Complete the Portal Workshop before this Code and SDK Workshop.  
This notebook recreates the baseline-agent capability with an independent `<PARTICIPANT_ID>-ws-...` Agent.

# Configuration

Load participant-scoped configuration, this step is purely to establish a deterministic agent name.   
The configuration modules (`workshop_core....`) are not used to create the agent - this is true for all notebooks in this workshop.

In [ ]:
# Load values from the local .env file before reading the workshop configuration.
from dotenv import load_dotenv

# Import the helpers that validate the participant ID and build a stable SDK name.
from workshop_core.config import WorkshopConfig
from workshop_core.naming import build_resource_name, sdk_participant_id

# Read the participant configuration and add the SDK-only -ws namespace.
load_dotenv()
config = WorkshopConfig.load()
sdk_participant = sdk_participant_id(config.participant_id)

# Build the name that the Foundry agent-version API will receive.
agent_name = build_resource_name(sdk_participant, "agent", "baseline")
print(f"Portal participant ID: {config.participant_id}")
print(f"SDK participant ID: {sdk_participant}")
print(f"SDK agent name: {agent_name}")

# Create the agent

Create a prompt-agent using the Foundry SDK.

In [ ]:
# Read the configured Foundry endpoint and model deployment from the environment.
import os

# Import the Foundry project client, prompt-agent definition, and Azure Identity credential provider.
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
from azure.identity import DefaultAzureCredential

PROJECT_ENDPOINT = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
MODEL_DEPLOYMENT_NAME = os.environ["MODEL_DEPLOYMENT_NAME"]

# DefaultAzureCredential supplies tokens from az login or another configured Azure identity source.
credential = DefaultAzureCredential()

# AIProjectClient manages Foundry project resources, including prompt-agent versions.
project = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)

# The OpenAI-compatible client performs runtime conversation and response operations for this project.
openai = project.get_openai_client()

# Define the baseline behavior that corresponds to the portal instructions field.
instructions = (
    "You are the Contoso SupportHub X1 support assistant. "
    "Answer only from connected sources when they are available. "
    "If a source does not contain the answer, say so. "
    "Never invent customer or private information."
)
test_prompt = "What is the warranty period for SupportHub X1?"

# Create the first version of the participant-scoped agent in the Foundry project.
baseline_agent = project.agents.create_version(
    agent_name=agent_name,
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT_NAME,
        instructions=instructions,
    ),
)
print(f"Agent identifier: {baseline_agent.id}\n")

# Send the test prompt to the Foundry runtime and select the agent by name.
baseline_response = openai.responses.create(
    input=test_prompt,
    extra_body={
        "agent_reference": {
            "name": baseline_agent.name,
            "type": "agent_reference",
        }
    },
)

print(f"Agent response:\n")
print(baseline_response.output_text)

# Inspect the prompt agent in Foundry

Open the Foundry portal and go to **Agents**.   
Locate the prompt agent using the SDK agent name printed above, then open it to inspect its version, model deployment, and instructions.  
This is the same persisted agent definition that the SDK created before invoking it.

# Test the agent

The first `responses.create()` call did not provide a `conversation` ID.  Foundry can still persist and display the resulting response or conversation record in the portal.  
The important point is that this request did not tell Foundry to reuse a specific earlier conversation as input context.

This next invocation explicitly creates a Foundry-managed conversation and passes its ID to `responses.create()`.   
When you reuse that ID on later calls, Foundry prepends the existing conversation items to the new request and appends the new input and output afterward.   
That explicit ID is what gives your code the same multi-turn context.

Invoke the created agent through the project runtime and examine the response.

In [ ]:
# Create a conversation so later responses can share a managed conversation context.
conversation = openai.conversations.create()

# Invoke the same agent with the existing test prompt in that conversation.
baseline_response = openai.responses.create(
    conversation=conversation.id,
    input=test_prompt,
    extra_body={
        "agent_reference": {
            "name": baseline_agent.name,
            "type": "agent_reference",
        }
    },
)

# Copy this exact ID when locating the conversation in Foundry Traces.
print(f"Conversation ID for Foundry Traces: {conversation.id}\n")

# Display the final text returned by the Foundry runtime.
print(baseline_response.output_text)

# Inspect the conversation in Foundry Traces

Open **Traces** in the Foundry portal and locate the conversation using the `Conversation ID for Foundry Traces` printed above.  
Open the matching trace to inspect the agent invocation, the input and output, and the conversation context that Foundry retained for this runtime call.

Retain `conversation.id` in application code when you want to continue this conversation in a later request. `baseline_response.id` identifies this individual response.

# Challenge

Change the agent instructions so the response begins with `SupportHub:` while preserving the rule against invented information. Create a new agent version, rerun the same prompt, and compare the version identifiers.

In [ ]:
## Code cell for the challenge, write your code below this line.

# Example solution - spoilers to the challenge

This solution creates a new version under the same agent name, adds the requested response prefix, and checks the result with the original prompt.

In [ ]:
# Extend the baseline instructions with the response-format requirement.
challenge_instructions = (
    instructions
    + " Begin every response with 'SupportHub:' while preserving all other rules."
)

# Create a new version of the same participant-scoped agent.
challenge_agent = project.agents.create_version(
    agent_name=agent_name,
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT_NAME,
        instructions=challenge_instructions,
    ),
)

# Run the original prompt against the new version and verify the requested format.
challenge_response = openai.responses.create(
    input=test_prompt,
    extra_body={
        "agent_reference": {
            "name": challenge_agent.name,
            "type": "agent_reference",
        }
    },
)
assert challenge_response.output_text.startswith("SupportHub:")
print(f"Baseline version: {baseline_agent.version}")
print(f"Challenge version: {challenge_agent.version}")
print(challenge_response.output_text)